In [6]:
import pandas as pd

datos = {
    'nombre': ['Ferretería García','Bar El Rincón','Peluquería Ana','Tienda Moda Sol','Gestoría Pérez', 'Clínica Dental Martínez','Academia Idiomas Sol'],
    'sector': ['Ferretería','Hostelería','Peluquería','Moda','Gestoría','Salud','Educación'],
    'facturacion': [320000, 180000, 95000, 210000, 140000, 280000,160000],
    'coste_ventas': [210000, 95000, 38000, 140000, 45000,84000,48000],
    'empleados': [5, 4, 2, 3, 3,6,5],
    'deuda': [45000, 12000, 8000, 35000, 5000,90000,15000],
    'activo_total': [180000, 85000, 40000, 120000, 70000,220000,75000],
    'año_fundacion': [2008, 2015, 2019, 2012, 2006,2014,2016]
}

df = pd.DataFrame(datos)
print(df)
print('\nNúmero de empresas analizadas:', len(df))

# --- BLOQUE 2: CÁLCULO DE RATIOS ---
# pandas calcula el ratio para TODAS las empresas a la vez con una sola línea
# es lo mismo que hacías con el diccionario pero para las 5 empresas simultáneamente

# Margen bruto en porcentaje
df['margen_bruto_pct'] = ((df['facturacion'] - df['coste_ventas']) / df['facturacion'] * 100).round(1)

# Ratio de endeudamiento (deuda / lo que tiene la empresa)
df['ratio_endeudamiento'] = (df['deuda'] / df['activo_total']).round(2)

# Facturación por empleado — eficiencia del negocio
df['fact_por_empleado'] = (df['facturacion'] / df['empleados']).round(0)

# Años que lleva abierta la empresa
df['años_activo'] = 2026 - df['año_fundacion']

# Ver la tabla con los nuevos ratios calculados
print(df[['nombre', 'margen_bruto_pct', 'ratio_endeudamiento', 'fact_por_empleado', 'años_activo']])

# --- BLOQUE 3: FILTROS DE INVERSIÓN ---
# Usamos los ratios calculados arriba para filtrar empresas
# df[condición] devuelve solo las filas donde la condición es True

# Empresas con bajo riesgo de deuda (menos del 30%)
bajo_riesgo = df[df['ratio_endeudamiento'] < 0.3]
print('Empresas con bajo endeudamiento:')
print(bajo_riesgo[['nombre', 'ratio_endeudamiento']])

# Empresas con buen margen Y bajo riesgo a la vez
# & significa que las dos condiciones tienen que cumplirse
candidatas = df[(df['margen_bruto_pct'] > 25) & (df['ratio_endeudamiento'] < 0.4)]
print('\nMejores candidatas para inversión:')
print(candidatas[['nombre', 'margen_bruto_pct', 'ratio_endeudamiento']])

# Ordenar todas las empresas por margen de mayor a menor
# ascending=False significa de mayor a menor
ranking = df.sort_values('margen_bruto_pct', ascending=False)
print('\nRanking por margen bruto:')
print(ranking[['nombre', 'margen_bruto_pct']])

# Empresas que están por encima de la media del grupo
media_margen = df['margen_bruto_pct'].mean()
sobre_media = df[df['margen_bruto_pct'] > media_margen]
print('\nEmpresas sobre la media del grupo:')
print(sobre_media['nombre'].tolist())

# --- BLOQUE 4: SCORING AUTOMÁTICO CON pd.cut() ---
# Versión mejorada del scoring de la Semana 1 — calcula para todas las empresas a la vez

# Scoring de endeudamiento
bins_deuda = [0, 0.25, 0.5, float('inf')]
labels_deuda = ['Bajo', 'Medio', 'Alto']
df['riesgo_deuda'] = pd.cut(df['ratio_endeudamiento'], bins=bins_deuda, labels=labels_deuda)

# Scoring de margen
bins_margen = [float('-inf'), 15, 30, float('inf')]
labels_margen = ['Bajo', 'Medio', 'Alto']
df['calidad_margen'] = pd.cut(df['margen_bruto_pct'], bins=bins_margen, labels=labels_margen)

# Puntuación numérica combinada
# Margen alto = 3 puntos, medio = 2, bajo = 1
# Deuda baja = 3 puntos, media = 2, alta = 1
# Más de 7 años activa = 2 puntos, menos = 1
mapa_margen = {'Bajo': 1, 'Medio': 2, 'Alto': 3}
mapa_deuda  = {'Bajo': 3, 'Medio': 2, 'Alto': 1}

df['puntos_margen']     = df['calidad_margen'].map(mapa_margen).astype(int)
df['puntos_deuda']      = df['riesgo_deuda'].map(mapa_deuda).astype(int)
df['puntos_antiguedad'] = df['años_activo'].apply(lambda x: 2 if x >= 7 else 1)
df['puntuacion_total']  = df['puntos_margen'] + df['puntos_deuda'] + df['puntos_antiguedad']

print(df[['nombre', 'calidad_margen', 'riesgo_deuda', 'años_activo', 'puntuacion_total']])

# --- BLOQUE 5: RANKING DE OPORTUNIDADES ---
# Ordenar todas las empresas por puntuación total de mayor a menor
# reset_index(drop=True) reinicia el índice desde 0 tras ordenar
ranking = df.sort_values('puntuacion_total', ascending=False).reset_index(drop=True)

# Empezar el índice en 1 en lugar de 0 — más legible como ranking
ranking.index = ranking.index + 1

# Columnas que aparecen en el ranking final
cols_ranking = ['nombre', 'sector', 'facturacion', 'margen_bruto_pct',
                'ratio_endeudamiento', 'años_activo', 'puntuacion_total',
                'calidad_margen', 'riesgo_deuda']

print('=' * 55)
print('RANKING DE OPORTUNIDADES DE INVERSIÓN EN PYMEs')
print('=' * 55)
print(ranking[cols_ranking].to_string())
print('\nMejor oportunidad:', ranking.iloc[0]['nombre'])
print('Mayor riesgo:     ', ranking.iloc[-1]['nombre'])

# --- BLOQUE 6: RESUMEN ESTADÍSTICO Y AGRUPACIÓN ---

# describe() da un resumen estadístico completo de todas las columnas numéricas
# muestra mínimo, máximo, media, mediana y desviación estándar de golpe
print('=== RESUMEN ESTADÍSTICO ===')
print(df[['margen_bruto_pct', 'ratio_endeudamiento', 'fact_por_empleado']].describe())

# Media de cada ratio para todo el conjunto
# sirve para saber si una empresa está por encima o por debajo de la media del grupo
print('\n=== MEDIAS DEL GRUPO ===')
print('Margen bruto medio:', df['margen_bruto_pct'].mean().round(1), '%')
print('Endeudamiento medio:', df['ratio_endeudamiento'].mean().round(2))
print('Facturación por empleado media:', df['fact_por_empleado'].mean().round(0), '€')

# Qué empresa tiene el mejor y peor margen
# idxmax() devuelve el índice de la fila con el valor más alto
print('\n=== DESTACADOS ===')
print('Mejor margen:', df.loc[df['margen_bruto_pct'].idxmax(), 'nombre'])
print('Peor margen:', df.loc[df['margen_bruto_pct'].idxmin(), 'nombre'])
print('Menor riesgo de deuda:', df.loc[df['ratio_endeudamiento'].idxmin(), 'nombre'])

# groupby agrupa las filas por sector y calcula la media de cada grupo
# útil para comparar sectores entre sí — la base del análisis sectorial
print('\n=== MEDIAS POR SECTOR ===')
por_sector = df.groupby('sector')[['margen_bruto_pct', 'ratio_endeudamiento']].mean().round(2)
print(por_sector)
print('\nSectores ordenados por margen (mejor a peor):')
print(por_sector.sort_values('margen_bruto_pct', ascending=False))


# --- BLOQUE 7: RESUMEN DEL ANÁLISIS ---
# Resumen ejecutivo del notebook — lo que le mostrarías a un inversor
# en la primera página de un informe antes de que lea la tabla completa

print('=== ANALIZADOR DE PYMEs v0.1 — RESUMEN ===')

# Total de empresas en el DataFrame
print(f'Empresas analizadas: {len(df)}')

# Número de sectores distintos — nunique() cuenta valores únicos
print(f'Sectores cubiertos: {df["sector"].nunique()}')

# Mejor empresa — primera fila del ranking (mayor puntuación)
print(f'Mejor oportunidad: {ranking.iloc[0]["nombre"]} ({ranking.iloc[0]["puntuacion_total"]} puntos)')

# Peor empresa — última fila del ranking (menor puntuación)
print(f'Mayor riesgo: {ranking.iloc[-1]["nombre"]} ({ranking.iloc[-1]["puntuacion_total"]} puntos)')

# Margen bruto medio del conjunto
print(f'Margen bruto medio del grupo: {df["margen_bruto_pct"].mean().round(1)}%')

# Sector con mejor margen medio
print(f'Sector más rentable: {df.groupby("sector")["margen_bruto_pct"].mean().idxmax()}')


# --- BLOQUE 8: EXPORTACIÓN A EXCEL ---
!pip install openpyxl -q

import openpyxl

nombre_archivo = 'ranking_pymes_v01.xlsx'

with pd.ExcelWriter(nombre_archivo, engine='openpyxl') as writer:

    # Hoja 1: Ranking completo ordenado por puntuación
    ranking[cols_ranking].to_excel(writer, sheet_name='Ranking', index=True)

    # Hoja 2: Datos originales con todos los ratios calculados
    df.to_excel(writer, sheet_name='Datos', index=False)

print(f'Archivo guardado: {nombre_archivo}')
print('Hojas: Ranking, Datos')



                    nombre      sector  facturacion  coste_ventas  empleados  \
0        Ferretería García  Ferretería       320000        210000          5   
1            Bar El Rincón  Hostelería       180000         95000          4   
2           Peluquería Ana  Peluquería        95000         38000          2   
3          Tienda Moda Sol        Moda       210000        140000          3   
4           Gestoría Pérez    Gestoría       140000         45000          3   
5  Clínica Dental Martínez       Salud       280000         84000          6   
6     Academia Idiomas Sol   Educación       160000         48000          5   

   deuda  activo_total  año_fundacion  
0  45000        180000           2008  
1  12000         85000           2015  
2   8000         40000           2019  
3  35000        120000           2012  
4   5000         70000           2006  
5  90000        220000           2014  
6  15000         75000           2016  

Número de empresas analizadas: 7
     